# Merge min-qlora.zip → GGUF for Ollama

1. Runtime → GPU
2. Upload your downloaded `min-qlora.zip` when asked
3. Run all cells
4. Download `companion-min-q4_k_m.gguf`
5. On your PC: put it in `finetune/ollama/` and run `register_ollama.ps1`

In [ ]:
from google.colab import files
uploaded = files.upload()
assert any(name.endswith('.zip') for name in uploaded), 'Upload min-qlora.zip'
ZIP_NAME = next(name for name in uploaded if name.endswith('.zip'))
print('Using', ZIP_NAME)

In [ ]:
# peft needs torchao>=0.16; Colab often ships an older torchao
%pip -q uninstall -y torchao
%pip -q install -U "torchao>=0.16.0" "transformers" "peft" "accelerate" "sentencepiece" "protobuf>=5.26.1,<6" "gguf"
import peft, torchao
print("peft", peft.__version__, "torchao", torchao.__version__)


In [ ]:
import json
import shutil
import zipfile
from pathlib import Path
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
ADAPT = Path('/content/adapter')
MERGED = Path('/content/merged-hf')

if ADAPT.exists():
    shutil.rmtree(ADAPT)
ADAPT.mkdir(parents=True)
with zipfile.ZipFile(ZIP_NAME, 'r') as zf:
    zf.extractall(ADAPT)
adapter_dir = next(Path(p).parent for p in ADAPT.rglob('adapter_config.json'))
print('Adapter at', adapter_dir)

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True
)
model = PeftModel.from_pretrained(base, str(adapter_dir))
merged = model.merge_and_unload()

if MERGED.exists():
    shutil.rmtree(MERGED)
MERGED.mkdir(parents=True)
merged.save_pretrained(str(MERGED), safe_serialization=True)

# IMPORTANT: use official base tokenizer files for llama.cpp convert
# (merged save can write extra_special_tokens as a list → AttributeError: .keys)
tok_base = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
tok_base.save_pretrained(str(MERGED))

cfg_path = MERGED / 'tokenizer_config.json'
cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
est = cfg.get('extra_special_tokens')
if isinstance(est, list):
    # convert list → dummy dict so transformers/llama.cpp don't crash
    cfg['extra_special_tokens'] = {str(i): t for i, t in enumerate(est)}
    cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Fixed extra_special_tokens list → dict')
elif est is not None:
    print('extra_special_tokens type:', type(est))

print('Merged HF saved to', MERGED)
print('Files:', sorted(p.name for p in MERGED.iterdir())[:20])


In [ ]:
# Skip cmake/llama-quantize (very slow on Colab).
# convert_hf_to_gguf supports q8_0 directly — good enough for Ollama.
import json
import subprocess
from pathlib import Path

MERGED = Path('/content/merged-hf')
assert MERGED.exists(), 'Run merge cell first'

cfg_path = MERGED / 'tokenizer_config.json'
cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
if isinstance(cfg.get('extra_special_tokens'), list):
    cfg['extra_special_tokens'] = {str(i): t for i, t in enumerate(cfg['extra_special_tokens'])}
    cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Patched extra_special_tokens')
else:
    from transformers import AutoTokenizer
    AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', trust_remote_code=True).save_pretrained(str(MERGED))
    print('Restored base tokenizer')

!rm -rf /content/llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
# Converter-only deps. Ignore Colab pip conflict warnings.
%pip -q install "gguf>=0.10" "sentencepiece" "protobuf>=5.26.1,<6"

OUT = Path('/content/companion-min-q8_0.gguf')
if OUT.exists():
    OUT.unlink()

r = subprocess.run(
    [
        'python', '/content/llama.cpp/convert_hf_to_gguf.py',
        str(MERGED),
        '--outfile', str(OUT),
        '--outtype', 'q8_0',
        '--verbose',
    ],
    capture_output=True,
    text=True,
)
print((r.stderr or r.stdout or '')[-4000:])
assert r.returncode == 0 and OUT.exists(), 'convert failed — see log above'
print('OK:', OUT, 'MB:', round(OUT.stat().st_size / 1024 / 1024, 1))
print('Ignore pip dependency conflict warnings — they are unrelated.')


In [ ]:
from google.colab import files
from pathlib import Path

for p in [
    Path('/content/companion-min-q8_0.gguf'),
    Path('/content/companion-min-q4_k_m.gguf'),
    Path('/content/companion-min-f16.gguf'),
]:
    if p.exists():
        print('Downloading', p, 'MB:', round(p.stat().st_size / 1024 / 1024, 1))
        files.download(str(p))
        break
else:
    raise FileNotFoundError('No GGUF found. Re-run the convert cell.')
